<h3 style="color:#6FA8DC; font-weight:bold">01 — Outliers: Introduction & Complete Guide</h3>

This notebook starts the **Outliers** topic in Feature Engineering.

We will understand:
- What are Outliers?
- When is an Outlier dangerous?
- Effect of Outliers on ML algorithms
- How to treat Outliers
- How to detect Outliers
- Main techniques for Outlier Detection
- When each technique should be used
- Modern ML approach and important production considerations

<h5 style="color:#78B89A; font-weight:bold;">What are Outliers? → simple meaning</h5>

An **outlier** is a data point that is unusually far away from most of the other observations.

Example:

```text
Normal values → 20, 22, 21, 24, 23, 25
Outlier       → 150
```

The value `150` is very different from the rest.

But remember:

**An outlier is not automatically a mistake.**

It may be:
- a genuine rare observation
- a data-entry error
- a measurement error
- a special business case
- fraud/anomaly

<h5 style="color:#78B89A; font-weight:bold;">Outlier ≠ Error</h5>

This is one of the most important rules.

Example:

A company's employee salaries are:

```text
₹30k, ₹35k, ₹40k, ₹42k, ₹45k, ₹50k, ₹5 lakh
```

₹5 lakh looks like an outlier.

But it may be a genuine salary of a senior executive.

So we should **not blindly delete it**.

First ask:

> Is this value wrong, or is it a genuine rare observation?

<h5 style="color:#78B89A; font-weight:bold;">When is an Outlier dangerous? → depends on the problem</h5>

An outlier can become dangerous when it:

1. Is caused by incorrect data collection.
2. Is caused by a data-entry mistake.
3. Strongly changes statistical calculations.
4. Distorts the relationship between features.
5. Makes a model learn unusual patterns instead of general patterns.
6. Causes unstable predictions.

Example:

```text
Age = 21, 22, 24, 25, 23, 999
```

`999` is almost certainly an invalid data value.

This should be investigated and corrected/removed.

But:

```text
Age = 21, 22, 24, 25, 23, 80
```

`80` may be completely valid.

<h5 style="color:#78B89A; font-weight:bold;">Effect of Outliers on ML Algorithms → very important</h5>

Different algorithms react differently to outliers.

| Algorithm / Method | Effect of Outliers |
|---|---|
| Linear Regression | High impact |
| Logistic Regression | Can affect boundary |
| KNN | Can affect distances |
| K-Means | High impact |
| SVM | Can affect decision boundary |
| Decision Tree | Usually less sensitive |
| Random Forest | Usually less sensitive |
| Gradient Boosting | Usually relatively robust, but still can be affected |
| Naive Bayes | Depends on distribution assumptions |
| PCA | High impact |
| Mean | Highly sensitive |
| Median | Much more robust |

### Why?

Many algorithms use:
- distance
- mean
- variance
- covariance
- squared error

Outliers can heavily influence these quantities.

<h5 style="color:#78B89A; font-weight:bold;">Example → Mean vs Median</h5>

Suppose:

```python
10, 12, 11, 13, 12
```

The mean is close to the normal values.

Now add an extreme value:

```python
10, 12, 11, 13, 12, 1000
```

The mean moves heavily toward `1000`.

The median changes much less.

This is why the median is called a **robust statistic**.

In [ ]:
import numpy as np
import pandas as pd

data = pd.Series([10, 12, 11, 13, 12, 1000])

print("Mean:", data.mean())
print("Median:", data.median())

<h5 style="color:#78B89A; font-weight:bold;">Effect on Visualization → easy to notice</h5>

Outliers can compress the majority of observations in plots.

For example:

```text
Normal values: 10–20
Outlier:       1000
```

A histogram or boxplot may make the normal values look extremely compressed.

This is one reason **boxplots** are useful during EDA.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

data = pd.Series([10, 12, 11, 13, 12, 14, 15, 13, 11, 1000])

plt.figure(figsize=(8, 4))
sns.boxplot(x=data)
plt.title("Example of an Outlier")
plt.show()

<h5 style="color:#78B89A; font-weight:bold;">How to treat Outliers? → main options</h5>

There is no single correct treatment.

```text
                OUTLIER
                   ↓
        ┌──────────┼──────────┐
        ↓          ↓          ↓
      Keep       Remove      Transform
        │          │          │
   genuine       error       log etc.
        │
        ↓
     Cap / Winsorize
```

### Main approaches

1. **Keep it**
   - When it is genuine and meaningful.

2. **Remove it**
   - When it is clearly an error or invalid observation.

3. **Cap / Winsorize it**
   - Replace extreme values with a chosen upper/lower limit.

4. **Transform the feature**
   - Example: log transformation for strongly right-skewed data.

5. **Use a robust model/statistic**
   - Some models are naturally less sensitive to outliers.

6. **Create a separate indicator**
   - Sometimes being an extreme observation itself contains useful information.

<h5 style="color:#78B89A; font-weight:bold;">Treatment decision → simple flow</h5>

```text
              Detect unusual value
                       ↓
              Is it a data error?
                 ↙           ↘
               YES            NO
                ↓              ↓
        Correct / Remove    Is it useful?
                              ↙     ↘
                            YES      NO
                             ↓        ↓
                           Keep     Cap /
                                   Transform
```

**Important:** Never remove an outlier just because it makes the dataset look cleaner.

<h3 style="color:#6FA8DC; font-weight:bold">How to Detect Outliers?</h3>

Outlier detection methods depend on:
- number of features
- distribution of the data
- whether the variable is numerical
- whether the data is normally distributed
- whether domain knowledge is available
- whether the problem is univariate or multivariate

<h5 style="color:#78B89A; font-weight:bold;">1. Visualization-based detection</h5>

Useful plots:

- Boxplot
- Histogram
- KDE plot
- Scatter plot

These are especially useful during **EDA**.

Example:

```python
sns.boxplot(x=df["Age"])
```

A boxplot can visually show values lying far beyond the normal range.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

df = pd.DataFrame({
    "salary": np.concatenate([
        np.random.normal(50000, 7000, 100),
        [150000, 180000, 250000]
    ])
})

sns.boxplot(x=df["salary"])
plt.title("Salary Outlier Detection")
plt.show()

<h5 style="color:#78B89A; font-weight:bold;">2. IQR Method → very important</h5>

IQR = Interquartile Range.

```text
IQR = Q3 - Q1
```

Where:

- Q1 = 25th percentile
- Q3 = 75th percentile

Common rule:

```text
Lower Limit = Q1 - 1.5 × IQR
Upper Limit = Q3 + 1.5 × IQR
```

Any value outside these limits is considered a potential outlier.

### Why use IQR?

It is based on the middle 50% of the data and is therefore more robust than methods based directly on the mean and standard deviation.

In [ ]:
Q1 = df["salary"].quantile(0.25)
Q3 = df["salary"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = df[
    (df["salary"] < lower_limit) |
    (df["salary"] > upper_limit)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)
print("Number of potential outliers:", len(outliers))

<h5 style="color:#78B89A; font-weight:bold;">3. Z-Score Method</h5>

Z-score tells us how far a value is from the mean in terms of standard deviations.

Formula:

```text
Z = (x - μ) / σ
```

Where:

- `x` = observation
- `μ` = mean
- `σ` = standard deviation

A common rule is:

```text
|Z| > 3  → potential outlier
```

### Important

The Z-score method works best when the feature is approximately **normally distributed**.

For highly skewed data, blindly using Z-score may not be appropriate.

In [ ]:
from scipy.stats import zscore

df["z_score"] = zscore(df["salary"])

potential_outliers = df[np.abs(df["z_score"]) > 3]

print(potential_outliers)

<h5 style="color:#78B89A; font-weight:bold;">4. Percentile / Quantile Method</h5>

Instead of using a statistical rule, we can define extreme regions using percentiles.

Example:

```text
Below 1st percentile  → extreme low values
Above 99th percentile → extreme high values
```

This can be useful when the business/domain defines what should be considered extreme.

Example:

```python
lower = df["salary"].quantile(0.01)
upper = df["salary"].quantile(0.99)
```

Then values outside those limits can be investigated or capped.

In [ ]:
lower = df["salary"].quantile(0.01)
upper = df["salary"].quantile(0.99)

outliers_percentile = df[
    (df["salary"] < lower) |
    (df["salary"] > upper)
]

print("1% limit:", lower)
print("99% limit:", upper)
print("Potential outliers:", len(outliers_percentile))

<h5 style="color:#78B89A; font-weight:bold;">5. Domain / Business Rules</h5>

This is often the most important method in real projects.

Example:

```text
Age cannot realistically be 250
Percentage cannot be 150%
Human height cannot normally be 10 meters
```

If the domain gives a valid range, use that knowledge.

Example:

```python
df[df["age"] > 120]
```

### Important

Statistical methods detect **unusual** values.

Domain rules can detect **impossible** values.

These are not always the same thing.

<h3 style="color:#6FA8DC; font-weight:bold">Univariate vs Multivariate Outliers</h3>

<h5 style="color:#78B89A; font-weight:bold;">Univariate Outlier → one feature</h5>

A value looks unusual when we inspect one feature alone.

Example:

```text
Salary = 10 lakh
```

Maybe it looks unusual compared with the salary distribution.

Common methods:
- IQR
- Z-score
- Percentiles
- Boxplot

<h5 style="color:#78B89A; font-weight:bold;">Multivariate Outlier → combination of features</h5>

Sometimes a value is not unusual in any single feature but becomes unusual when features are considered together.

Example:

```text
Age = 25
Income = ₹2 crore
```

Each value may individually be possible, but the combination may be unusual for the dataset.

Multivariate methods include:
- Isolation Forest
- Local Outlier Factor (LOF)
- One-Class SVM
- clustering/distance-based approaches

These are more advanced and will be studied separately.

<h3 style="color:#6FA8DC; font-weight:bold">Main Outlier Detection Techniques</h3>

```text
OUTLIER DETECTION
│
├── Visual
│   ├── Boxplot
│   ├── Histogram
│   └── Scatter plot
│
├── Statistical / Univariate
│   ├── IQR
│   ├── Z-score
│   └── Percentile / Quantile
│
├── Domain Based
│   └── Business / physical limits
│
└── Multivariate / ML Based
    ├── Isolation Forest
    ├── LOF
    ├── One-Class SVM
    └── Distance / clustering methods
```

<h3 style="color:#6FA8DC; font-weight:bold">Which Method Should I Use?</h3>

| Situation | Useful starting method |
|---|---|
| Need quick EDA | Boxplot |
| Numerical, skewed data | IQR |
| Approximately normal data | Z-score |
| Business-defined extreme range | Percentiles / domain rules |
| Clearly impossible values | Domain rules |
| Multiple features interact | Multivariate methods |
| Large ML dataset | ML-based detection may be considered |

There is **no universal best outlier detector**.

The choice depends on the data and the objective.

<h3 style="color:#6FA8DC; font-weight:bold">Outliers and Production ML</h3>

During production, do not calculate outlier limits independently on every new batch.

A safer workflow is:

```text
Training Data
     ↓
EDA
     ↓
Learn outlier rule / thresholds
     ↓
Save preprocessing logic
     ↓
New Data
     ↓
Apply same learned rule
     ↓
Model
```

For example, if you use percentile capping, the limits should be learned from the **training data**, not from the complete dataset.

This prevents information from the validation/test/future data from leaking into training.

<h5 style="color:#78B89A; font-weight:bold;">Outlier treatment inside a Pipeline</h5>

For production ML, preprocessing should ideally be connected to the model through a `Pipeline`.

```python
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("preprocessing", preprocessing_step),
    ("model", model)
])

pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
```

This keeps training and prediction preprocessing consistent.

Some custom outlier-treatment operations require a custom transformer or an appropriate third-party/production preprocessing component rather than directly deleting rows inside a normal Pipeline.

<h3 style="color:#6FA8DC; font-weight:bold">Important Rules to Remember ⭐</h3>

1. **Outlier does not automatically mean error.**
2. Investigate before removing.
3. Domain knowledge is extremely important.
4. IQR is a strong general-purpose univariate method.
5. Z-score is more suitable when the distribution is approximately normal.
6. Mean and standard deviation are sensitive to outliers.
7. Median and IQR are more robust.
8. Linear Regression and KNN can be strongly affected.
9. Tree-based models are generally less sensitive.
10. Do not calculate preprocessing thresholds using test data.
11. Outlier treatment should be consistent between training and production.
12. Multivariate outliers require different techniques from simple IQR/Z-score detection.

<h3 style="color:#6FA8DC; font-weight:bold">Final Revision Flow</h3>

```text
                 OUTLIERS
                    ↓
        Unusually different value
                    ↓
             Is it an error?
              ↙          ↘
            YES           NO
             ↓             ↓
       Correct/Remove   Investigate
                           ↓
                    Keep / Cap /
                    Transform
                           ↓
                  Choose detection
                      technique
                           ↓
        ┌──────────┬───────────┬───────────┐
        ↓          ↓           ↓           ↓
      IQR       Z-score    Percentile    Domain
        │          │           │           │
        └──────────┴───────────┴───────────┘
                           ↓
                 If multiple features
                           ↓
              Multivariate ML methods
```

### One-line definition

> **An outlier is an observation that is unusually different from the majority of observations in a dataset.**

The important question is not simply **"Is this an outlier?"**

It is:

> **"Why is this value unusual, and what should I do with it?"**